# 🛡️ FlowRun Streamlet: IoC Triage — v0.0.32
## Automated Threat Intelligence Triage — Jupyter Notebook Interface

This notebook mirrors the full CLI agent functionality. Run cells 1–5 in order to set up the environment,
then use Cell 6 to submit IOCs for triage. Results appear inline, and the active OTLP trace destination is shown in Cell 8.

**Stack:** LangGraph + LangChain + OpenAI GPT-4o + OpenTelemetry (Traceloop)

---

### ⚠️ Virtual Environment Users
If you installed dependencies in a **venv**, you must register it as a Jupyter kernel first:

```bash
source .venv/bin/activate
pip install ipykernel
python -m ipykernel install --user --name=flowrun --display-name="FlowRun (venv)"
```

Then go to **Kernel → Change kernel → FlowRun (venv)** before running any cells.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 1 — Install & Import
# ═══════════════════════════════════════════════════════════════════════════
# Uncomment the line below to install dependencies on first run:
# !pip install -r requirements.txt ipykernel

import os
import asyncio
from agent.credentials import resolve_credentials
from agent.tracing import init_tracing
from agent.graph import build_graph
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print('✅ All imports successful.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 2 — API Key Setup
# ═══════════════════════════════════════════════════════════════════════════
# Option A: Interactive masked input (recommended for shared environments)
#   resolve_credentials() will use getpass() for any missing keys.
#
# Option B: Load from .env file (recommended for daily use)
#   Create a .env file from .env.template and fill in your keys.
#   No quotes around values — just paste the key after the = sign.
#
# NEVER assign keys as plain string literals in this cell.

resolve_credentials()
print('✅ All API keys resolved.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 3 — OpenTelemetry Tracing Init
# ═══════════════════════════════════════════════════════════════════════════
# Initialise the OpenTelemetry tracer. Spans are exported via OTLP/HTTP to
# the local collector agent at http://localhost:4318 by default. To ship to
# a different endpoint, set OTEL_EXPORTER_OTLP_ENDPOINT before running.
# If the collector is unreachable, tracing is skipped (triage continues normally).

endpoint = init_tracing()
if endpoint:
    print(f'✅ OpenTelemetry tracing initialised. Endpoint: {endpoint}')
else:
    print('⚠️ OpenTelemetry tracing unavailable — triage will continue without tracing.')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 4 — Tool Definitions
# ═══════════════════════════════════════════════════════════════════════════
# Instantiate all 5 LangChain Tool wrappers for threat intelligence APIs.

from agent.tools.virustotal import VirusTotalTool
from agent.tools.abuseipdb import AbuseIPDBTool
from agent.tools.otx import OTXTool
from agent.tools.urlscan import URLScanTool
from agent.tools.nvd import NVDTool

vt_tool       = VirusTotalTool()
abuseipdb_tool = AbuseIPDBTool()
otx_tool      = OTXTool()
urlscan_tool  = URLScanTool()
nvd_tool      = NVDTool()

print(f'✅ Tools ready: {[t.name for t in [vt_tool, abuseipdb_tool, otx_tool, urlscan_tool, nvd_tool]]}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 5 — Graph Compilation
# ═══════════════════════════════════════════════════════════════════════════
# Build and compile the LangGraph StateGraph.

graph = build_graph()
print('✅ LangGraph StateGraph compiled.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 6 — IOC Input Widget
# ═══════════════════════════════════════════════════════════════════════════
# Enter an IOC in the text field and click Analyze to run a full triage.
# Results render inline below; the active OTLP trace destination is shown in Cell 8.
#
# Test IOCs to try:
#   8.8.8.8                                    (clean IP — expect CLEAN/LOW)
#   44d88612fea8a8f36de82e1278abb02f            (EICAR MD5 — expect HIGH/CRITICAL)
#   malware.wicar.org                           (test domain — expect MEDIUM+)
#   CVE-2021-44228                              (Log4Shell — expect MEDIUM+)
#   npm:postmark-mcp                            (suspicious package — test OSV.dev)
#   pypi:requests                               (clean package — expect CLEAN/LOW)
#   traceroute                                   (bare name — scans all ecosystems)
#   npm:postmark-mcp                            (package — tests OSV + npm registry)

ioc_input = widgets.Text(
    placeholder='Enter IOC: IP, domain, URL, hash, or CVE...',
    layout=widgets.Layout(width='65%')
)
analyze_btn = widgets.Button(
    description='⚡ Analyze',
    button_style='danger',
    layout=widgets.Layout(width='120px')
)
output_area = widgets.Output()

async def on_click(b):
    with output_area:
        clear_output(wait=True)
        ioc = ioc_input.value.strip()
        if not ioc:
            display(HTML('<p style="color:red">⚠️ Please enter an IOC.</p>'))
            return
        display(HTML(f'<p>🔍 Triaging <b>{ioc}</b> — please wait...</p>'))
        try:
            result = await graph.ainvoke({'ioc_raw': ioc})
            clear_output(wait=True)
            display(HTML(result.get('report_html', '<p>No report generated.</p>')))
            endpoint = result.get('trace_endpoint', '')
            if endpoint:
                display(HTML(
                    f'<p style="color:#6B7280; font-size:0.9em;">'
                    f'📡 Spans exported to <code>{endpoint}</code></p>'
                ))
        except SystemExit as exc:
            display(HTML(f'<p style="color:orange">⚠️ {exc}</p>'))
        except Exception as exc:
            display(HTML(
                f'<p style="color:red">❌ Triage failed: '
                f'{type(exc).__name__}: {exc}</p>'
            ))

# Capture the ipykernel's running event loop at cell-execution time.
# Python 3.14 removed the implicit-loop fallback that asyncio.ensure_future()
# previously relied on inside synchronous ipywidgets callbacks.
try:
    _loop = asyncio.get_running_loop()
except RuntimeError:
    _loop = asyncio.get_event_loop()

def _schedule_triage(b):
    _loop.create_task(on_click(b))

analyze_btn.on_click(_schedule_triage)
display(widgets.VBox([widgets.HBox([ioc_input, analyze_btn]), output_area]))


## Cell 7 — Report Display

The threat report renders **inline** in Cell 6's output area above after you click **⚡ Analyze**.

The report includes:
- IOC summary with detected type
- Colour-coded severity badge (CLEAN / LOW / MEDIUM / HIGH / CRITICAL)
- Per-source intelligence findings
- Collapsible score breakdown
- Correlation summary and recommended actions

To triage a new IOC, simply clear the input field above, enter the new IOC, and click Analyze again.
You do **not** need to re-run Cells 1–5 unless you've restarted the kernel.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 8 — Trace Destination
# ═══════════════════════════════════════════════════════════════════════════
# Traces are shipped via OTLP/HTTP to the configured collector. Inspect them
# in whichever observability tool your collector is forwarding data to.

import os

endpoint = (
    os.getenv('OTEL_EXPORTER_OTLP_ENDPOINT')
    or os.getenv('TRACELOOP_BASE_URL')
    or 'http://localhost:4318'
)
print(f'Active OTLP endpoint: {endpoint}')
print('Service name: flowrun-streamlet-ioc-triage')
